# Triton Matmul — Colab Runner (thin)

This notebook contains **no** kernel, timing, or correctness logic. It only sets up the repo on the import path and loads the repository modules, so the exact same code runs here as anywhere else.

**Phase 0 only:** environment verification + a 256x256 smoke test with an explicit pass/fail tolerance check. Run the cells top to bottom and paste the output back into Kiro before proceeding.

**Import note:** `tests` is a common top-level package name and can collide with other packages on Colab's `sys.path`. To avoid that permanently, Cell 3 loads the test module **directly from its file path** via `importlib.util.spec_from_file_location` instead of `import tests.test_matmul`.

## Cell 1 — Environment verification
Confirms the runtime matches the confirmed stack (Tesla T4, CC 7.5, PyTorch 2.11.0+cu130, CUDA 13.0, Triton 3.6.0). Do **not** reinstall or downgrade torch/triton.

In [ ]:
import torch, triton
print('torch        :', torch.__version__)
print('triton       :', triton.__version__)
print('cuda (torch) :', torch.version.cuda)
print('cuda avail   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu          :', torch.cuda.get_device_name(0))
    cc = torch.cuda.get_device_capability(0)
    print('compute cap  :', f'{cc[0]}.{cc[1]}')

## Cell 2 — Repo setup (pick ONE option)
Get the repository onto Colab's import path. Use **either** the git clone option **or** the upload/Drive option. Set `REPO_ROOT` to the repository root so later cells can locate module files reliably.

In [ ]:
import sys, os

# ---- Option A: clone from GitHub ------------------------------------------
# !git clone https://github.com/anshikachourey/triton-gpu-kernels.git
# REPO_ROOT = '/content/triton-gpu-kernels'

# ---- Option B: upload the folder or mount Google Drive --------------------
# from google.colab import drive; drive.mount('/content/drive')
# REPO_ROOT = '/content/drive/MyDrive/<path-to-repo>'

# Set REPO_ROOT above (uncomment one option), then:
assert 'REPO_ROOT' in globals(), 'Set REPO_ROOT to the repository root first.'
REPO_ROOT = os.path.abspath(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# The kernel module is a clean package import (kernels is an uncommon name).
from kernels.matmul import MatmulConfig, matmul
print('REPO_ROOT    :', REPO_ROOT)
print('kernels.matmul imported OK')

## Cell 3 — Phase 0 smoke test (256x256) with explicit tolerance check
Loads the test module **directly from its file path** (avoids the `tests` package-name collision), runs the custom kernel vs `torch.matmul`, and applies an explicit elementwise `torch.testing.assert_close` with `atol=1e-2`, `rtol=1e-2` (compared in FP32). If `tl.dot` were unsupported this would raise during the run — paste the full traceback if so.

In [ ]:
import importlib.util, os

test_path = os.path.join(REPO_ROOT, 'tests', 'test_matmul.py')
print('Test file exists:', os.path.exists(test_path))

# Load tests/test_matmul.py directly by file path to avoid any `tests`
# top-level package collision on Colab's sys.path.
spec = importlib.util.spec_from_file_location('phase0_test_matmul', test_path)
test_matmul = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_matmul)

out = test_matmul.smoke_test_256()
print('Smoke test 256x256')
print('  output shape         :', out['shape'])
print('  output dtype         :', out['dtype'])
print('  config (BM,BN,BK,W,S):', out['config'])
print('  max abs error (FP32) : %.6e' % out['max_abs_error_fp32eval'])
print('  robust rel error     : %.6e' % out['robust_rel_error_fp32eval'])
print('  tolerance (atol,rtol): (%s, %s)' % (out['atol'], out['rtol']))
print('  elementwise check    :', out['check_message'])
print('  passed               :', out['passed'])

## Cell 4 — Phase 1 + Phase 2: full correctness suite
Runs the custom kernel across the required shapes (square, rectangular, small non-square, non-multiple-of-tile) and all six fixed configs (num_stages=2), comparing to PyTorch FP16 (primary, atol=rtol=1e-2) with FP32 diagnostic errors recorded too. Writes `results/correctness.csv` and prints a per-row PASS/FAIL/error summary.

Reuses the file-path module loaded in Cell 3 (no `tests` package collision). Tolerances are fixed and never loosened; actual compilation/unsupported/numerical failures are recorded, not hidden. **No benchmarking or plotting here.**

In [ ]:
# `test_matmul` was loaded by file path in Cell 3. Reload defensively in case
# this cell is run on its own.
import importlib.util, os
if 'test_matmul' not in globals():
    _p = os.path.join(REPO_ROOT, 'tests', 'test_matmul.py')
    _spec = importlib.util.spec_from_file_location('phase0_test_matmul', _p)
    test_matmul = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(test_matmul)

records = test_matmul.run_correctness()
csv_path = test_matmul.write_correctness_csv(records, os.path.join(REPO_ROOT, 'results', 'correctness.csv'))
summary = test_matmul.summarize(records)
print('wrote:', csv_path)
print('summary:', summary)
print()
for r in records:
    tag = r.status if r.status == 'error' else ('PASS' if r.passed else 'FAIL')
    print('[%s] %-32s cfg=(%d,%d,%d,%d,%d) max_abs_fp16=%s max_abs_fp32=%s %s' % (
        tag, r.label, r.block_m, r.block_n, r.block_k, r.num_warps, r.num_stages,
        r.max_abs_error_fp16, r.max_abs_error_fp32, r.error_reason))

## Cell 5 — Diagnostic: investigate the rectangular-case failure (1000x1500x700)
Investigation only. Uses the **same** deterministic inputs (seed 2) and the **same** fixed tolerances (atol=rtol=1e-2). Does **not** loosen tolerances, change the kernel, or alter input generation. Reports: violation count/percentage, max excess over tolerance, worst 10 mismatching coordinates (Triton / FP16-ref / FP32-ref / abs diff / allowed tol), boundary-vs-interior clustering, and how much the PyTorch FP16 reference disagrees with the FP32 reference under the same rule.

In [ ]:
import importlib.util, os
if 'test_matmul' not in globals():
    _p = os.path.join(REPO_ROOT, 'tests', 'test_matmul.py')
    _spec = importlib.util.spec_from_file_location('phase0_test_matmul', _p)
    test_matmul = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(test_matmul)

from kernels.matmul import MatmulConfig
d = test_matmul.diagnose_case(1000, 1500, 700, seed=2,
                              config=MatmulConfig(64, 64, 32, 4, 2), top=10)
test_matmul.print_diagnosis(d)

## Cell 6 — Targeted FP32-reference verification (rectangular case)
Checks the FULL 1000x1500x700 Triton output against the **FP32** reference using the same fixed rule (atol=rtol=1e-2). Does **not** relabel the existing FP16-reference results or loosen tolerances. If this passes, the earlier FP16-reference failures are a reference-precision artifact and Phase 3 can proceed.

In [ ]:
import importlib.util, os
if 'test_matmul' not in globals():
    _p = os.path.join(REPO_ROOT, 'tests', 'test_matmul.py')
    _spec = importlib.util.spec_from_file_location('phase0_test_matmul', _p)
    test_matmul = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(test_matmul)

from kernels.matmul import MatmulConfig
v = test_matmul.verify_against_fp32(1000, 1500, 700, seed=2,
                                    config=MatmulConfig(64, 64, 32, 4, 2))
test_matmul.print_fp32_verification(v)

## Cell 7 — Phase 3: benchmarking framework
Benchmarks the identical operation C = A @ B with **preallocated FP16 output on both sides**, explicit `do_bench(return_mode='median')` primary latency, separate p20/p50/p80 quantiles, TFLOPS, and Triton-vs-torch speedup from measured medians. Attempts every size x config; OOM / OutOfResources / compile failures are recorded as skipped rows with the actual error message. Writes `results/benchmarks.csv`.

Sizes: 512^3, 1024^3, 2048^3, 4096^3, 1000x1500x700. Expect 4096^3 and/or the 128x128 configs to possibly skip on the T4 — that is a valid finding, not a failure to hide.

In [ ]:
import importlib.util, os
_bp = os.path.join(REPO_ROOT, 'benchmarks', 'benchmark.py')
_bspec = importlib.util.spec_from_file_location('phase3_benchmark', _bp)
benchmark = importlib.util.module_from_spec(_bspec)
_bspec.loader.exec_module(benchmark)

print('env:', benchmark.environment_info())
records = benchmark.run_benchmarks()
csv_path = benchmark.write_csv(records, os.path.join(REPO_ROOT, 'results', 'benchmarks.csv'))
print('wrote:', csv_path)
print()
benchmark.print_benchmarks(records)

## Cell 8 — Methodology review (resources, clocks, repeatability)
Does NOT overwrite the first benchmark run. Inspects compiled-kernel resource metadata (registers / shared memory / spills, where Triton exposes them — reported raw, no spilling claim), snapshots GPU clock/temperature, and re-measures 64x64x32 at 4 vs 8 warps on 1024^3 and 4096^3 over several repeats to confirm the observed speedups and compare primary-median vs quantile-p50. Writes `results/benchmarks_repeat.csv` (separate file).

In [ ]:
import importlib.util, os
if 'benchmark' not in globals():
    _bp = os.path.join(REPO_ROOT, 'benchmarks', 'benchmark.py')
    _bspec = importlib.util.spec_from_file_location('phase3_benchmark', _bp)
    benchmark = importlib.util.module_from_spec(_bspec)
    _bspec.loader.exec_module(benchmark)

from kernels.matmul import MatmulConfig

print('GPU state snapshot:', benchmark.gpu_state())
print()
print('Kernel resource metadata (raw; no spilling claim):')
for cfg in [MatmulConfig(64,64,32,4,2), MatmulConfig(64,64,32,8,2),
            MatmulConfig(128,128,32,4,2), MatmulConfig(128,128,32,8,2)]:
    print(' ', benchmark.kernel_resources(1024,1024,1024, config=cfg))
print()
rows = benchmark.repeat_compare(repeats=5)
rpath = benchmark.write_repeat_csv(rows, os.path.join(REPO_ROOT, 'results', 'benchmarks_repeat.csv'))
print('wrote:', rpath)
print()
benchmark.print_repeat_summary(rows)

## Cell 9 — Phase 5 plots (from measured CSV only)
Generates `results/latency_comparison.png` and `results/throughput_comparison.png` from the measured `results/benchmarks.csv` (fastest custom config vs PyTorch). Reads observed data only — nothing fabricated.

In [ ]:
import importlib.util, os
if 'benchmark' not in globals():
    _bp = os.path.join(REPO_ROOT, 'benchmarks', 'benchmark.py')
    _bspec = importlib.util.spec_from_file_location('phase3_benchmark', _bp)
    benchmark = importlib.util.module_from_spec(_bspec)
    _bspec.loader.exec_module(benchmark)

csv_path = os.path.join(REPO_ROOT, 'results', 'benchmarks.csv')
lat = benchmark.plot_latency(csv_path, os.path.join(REPO_ROOT, 'results', 'latency_comparison.png'), log_y=True)
thr = benchmark.plot_throughput(csv_path, os.path.join(REPO_ROOT, 'results', 'throughput_comparison.png'))
spd = benchmark.plot_config_speedup(csv_path, os.path.join(REPO_ROOT, 'results', 'config_speedup.png'), baseline_warps=4, compare_warps=8)
print('wrote:', lat)
print('wrote:', thr)
print('wrote:', spd)
from IPython.display import Image, display
display(Image(lat)); display(Image(thr)); display(Image(spd))

## Cell 10 — Hardware diagnostics (read-only)
Isolated hardware investigation. Launches `_matmul_kernel[grid](...)` directly (does NOT touch the public `matmul()` wrapper) for the 64x64/8-warp and 128x128/4-warp configs, prints the compiled object's public attributes first, then register/spill/shared metadata, PTX (mma.sync / hmma / fma counts + examples), TTGIR dot-lowering, and a cuobjdump fallback if available. Writes `results/diagnostics/`. **Draws no conclusion about Tensor Core use or bottlenecks** — review the saved PTX/TTGIR/SASS first.

In [ ]:
import importlib.util, os
_dp = os.path.join(REPO_ROOT, 'benchmarks', 'diagnostics.py')
_dspec = importlib.util.spec_from_file_location('diagnostics', _dp)
diagnostics = importlib.util.module_from_spec(_dspec)
_dspec.loader.exec_module(diagnostics)

diag_dir = os.path.join(REPO_ROOT, 'results', 'diagnostics')
hw = diagnostics.run_hardware_diagnostics(out_dir=diag_dir)
diagnostics.print_hardware_summary(hw)

## Cell 11 — Numerical diagnostics (read-only)
Compares the custom Triton output on 1000x1500x700 (original input scheme, seed 2) against PyTorch FP16, PyTorch FP16 with `allow_fp16_reduced_precision_reduction=False` (if supported; restored in finally), and an FP64 reference rounded to FP16. Reports max abs error, relative-error stats, tolerance violations, FP16-ULP error, and the count of Triton elements exactly matching the rounded FP64 result; plus a `torch.equal` cross-config check. Writes `results/diagnostics/`; does NOT touch `results/correctness.csv`.

In [ ]:
import importlib.util, os
if 'diagnostics' not in globals():
    _dp = os.path.join(REPO_ROOT, 'benchmarks', 'diagnostics.py')
    _dspec = importlib.util.spec_from_file_location('diagnostics', _dp)
    diagnostics = importlib.util.module_from_spec(_dspec)
    _dspec.loader.exec_module(diagnostics)

diag_dir = os.path.join(REPO_ROOT, 'results', 'diagnostics')
num = diagnostics.run_numerical_diagnostics(config=diagnostics.CONFIG_64, out_dir=diag_dir)
eq = diagnostics.run_cross_config_equality(out_dir=diag_dir)
diagnostics.print_numerical_summary(num, eq)

## Cell 12 — CUDA C++ WMMA extension: toolchain check + compile
Separate extension (does NOT touch the Triton kernel). Prints the toolchain versions first, then compiles the naive WMMA kernel to a shared library with `nvcc -arch=sm_75`. **No installs, no cpp_extension.** If `nvcc` is missing or the compile fails, stop and paste the output.

In [ ]:
import subprocess, os, torch
print('=== nvcc --version ==='); print(subprocess.run(['nvcc','--version'], capture_output=True, text=True).stdout or 'nvcc NOT FOUND')
print('=== nvidia-smi ==='); print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:1200])
print('torch.version.cuda =', torch.version.cuda)

src = os.path.join(REPO_ROOT, 'cuda', 'wmma_matmul.cu')
so  = os.path.join(REPO_ROOT, 'cuda', 'libwmma_matmul.so')
cmd = ['nvcc', '-arch=sm_75', '-Xcompiler', '-fPIC', '-shared', src, '-o', so]
print('\n=== compile ===\n', ' '.join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print('returncode:', proc.returncode)
print(proc.stdout); print(proc.stderr)
print('library exists:', os.path.exists(so))

## Cell 13 — CUDA WMMA correctness (vs FP64 reference and vs Triton)
Loads the `.so` via ctypes and feeds the SAME torch FP16 inputs to the CUDA, Triton, and PyTorch paths. Reports max abs error, tolerance violations, and exact-match counts vs the FP64→FP16 reference, plus whether CUDA matches Triton bitwise. 256×256 and 1024×1024 only.

In [ ]:
import importlib.util, os
_rp = os.path.join(REPO_ROOT, 'cuda', 'run_wmma.py')
_rspec = importlib.util.spec_from_file_location('run_wmma', _rp)
run_wmma = importlib.util.module_from_spec(_rspec)
_rspec.loader.exec_module(run_wmma)

lib = run_wmma.load_library(os.path.join(REPO_ROOT, 'cuda', 'libwmma_matmul.so'))
diag = os.path.join(REPO_ROOT, 'results', 'diagnostics', 'cuda')
recs = run_wmma.correctness(lib, sizes=(256, 1024), out_dir=diag)
run_wmma.print_correctness(recs)

## Cell 14 — CUDA SASS inspection (cuobjdump)
Disassembles the compiled `.so` and reports Tensor Core **HMMA** instruction names + counts, plus FFMA, LDL/STL, registers, spills, and shared memory. Saves `-sass` and `--dump-resource-usage` output to `results/diagnostics/cuda/`. **Presence of WMMA source alone does not prove Tensor Core execution — judge from the SASS.**

In [ ]:
import subprocess, os, re
so  = os.path.join(REPO_ROOT, 'cuda', 'libwmma_matmul.so')
diag = os.path.join(REPO_ROOT, 'results', 'diagnostics', 'cuda')
os.makedirs(diag, exist_ok=True)

res = subprocess.run(['cuobjdump', '--dump-resource-usage', so], capture_output=True, text=True)
open(os.path.join(diag, 'cuda_res_usage.txt'), 'w').write(res.stdout + res.stderr)
sass = subprocess.run(['cuobjdump', '-sass', so], capture_output=True, text=True)
open(os.path.join(diag, 'cuda_sass.txt'), 'w').write(sass.stdout + sass.stderr)

text = sass.stdout
# Report ACTUAL instruction opcode names (first token after optional predicate).
from collections import Counter
ops = Counter()
for ln in text.splitlines():
    m = re.search(r'/\*[0-9a-f]+\*/\s+(@!?P\d+\s+)?([A-Z][A-Z0-9_.]*)', ln)
    if m: ops[m.group(2).split('.')[0]] += 1
hmma = {k: v for k, v in ops.items() if 'HMMA' in k or 'IMMA' in k or 'MMA' in k}
print('resource usage head:'); print('\n'.join(res.stdout.splitlines()[:20]))
print('\nMMA-family opcodes :', hmma if hmma else 'NONE FOUND')
print('FFMA count         :', ops.get('FFMA', 0))
print('LDL / STL          :', ops.get('LDL', 0), '/', ops.get('STL', 0))
print('top opcodes        :', ops.most_common(12))
print('saved SASS + res-usage to', diag)

## Cell 15 — CUDA WMMA performance (CUDA vs Triton vs PyTorch)
Interleaved, warmed-up, synchronized timing with preallocated FP16 output on all three paths; captures SM clock per repeat. Writes `results/diagnostics/cuda/cuda_performance.json`. Results are reported separately from the original benchmark data; no claims until reviewed.

In [ ]:
import importlib.util, os
if 'run_wmma' not in globals():
    _rp = os.path.join(REPO_ROOT, 'cuda', 'run_wmma.py')
    _rspec = importlib.util.spec_from_file_location('run_wmma', _rp)
    run_wmma = importlib.util.module_from_spec(_rspec)
    _rspec.loader.exec_module(run_wmma)
    lib = run_wmma.load_library(os.path.join(REPO_ROOT, 'cuda', 'libwmma_matmul.so'))

diag = os.path.join(REPO_ROOT, 'results', 'diagnostics', 'cuda')
rows = run_wmma.performance(lib, sizes=(256, 1024), repeats=5, out_dir=diag)
run_wmma.print_performance(rows)

## Cell 16 — Optimized (shared-memory) CUDA WMMA: compile
Separate second kernel; the naive baseline (`wmma_matmul.cu`) is untouched. Compiles `cuda/wmma_matmul_shared.cu` with `nvcc -arch=sm_75` into its own `.so`. No installs. If it fails, stop and paste the output (time-box).

In [ ]:
import subprocess, os
src = os.path.join(REPO_ROOT, 'cuda', 'wmma_matmul_shared.cu')
so  = os.path.join(REPO_ROOT, 'cuda', 'libwmma_matmul_shared.so')
cmd = ['nvcc', '-arch=sm_75', '-Xcompiler', '-fPIC', '-shared', src, '-o', so]
print(' '.join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print('returncode:', proc.returncode)
print(proc.stdout); print(proc.stderr)
print('library exists:', os.path.exists(so))

## Cell 17 — Shared vs naive vs Triton vs PyTorch: correctness
Same inputs for all paths. Reports shared-kernel max abs error, tolerance violations, and exact-match % vs the FP64→FP16 reference, plus whether the shared kernel matches the naive baseline bitwise. 256 and 1024.

In [ ]:
import importlib.util, os, sys
cuda_dir = os.path.join(REPO_ROOT, 'cuda')
if cuda_dir not in sys.path: sys.path.insert(0, cuda_dir)  # so run_wmma_shared can import run_wmma

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod); return mod

run_wmma = _load('run_wmma', os.path.join(cuda_dir, 'run_wmma.py'))
sys.modules['run_wmma'] = run_wmma
rws = _load('run_wmma_shared', os.path.join(cuda_dir, 'run_wmma_shared.py'))

naive_lib = run_wmma.load_library(os.path.join(cuda_dir, 'libwmma_matmul.so'))
shared_lib = rws.load_shared_library(os.path.join(cuda_dir, 'libwmma_matmul_shared.so'))
diag = os.path.join(REPO_ROOT, 'results', 'diagnostics', 'cuda')
recs = rws.correctness(naive_lib, shared_lib, sizes=(256, 1024), out_dir=diag)
rws.print_correctness(recs)

## Cell 18 — Shared kernel SASS inspection (cuobjdump)
Disassembles the shared-memory `.so`: HMMA names/counts, FFMA, LDL/STL, and reports registers / shared memory / spills via `--dump-resource-usage`. Saves to `results/diagnostics/cuda/`. Presence of WMMA source alone does not prove Tensor Core use — judge from the SASS.

In [ ]:
import subprocess, os, re
from collections import Counter
so  = os.path.join(REPO_ROOT, 'cuda', 'libwmma_matmul_shared.so')
diag = os.path.join(REPO_ROOT, 'results', 'diagnostics', 'cuda')
os.makedirs(diag, exist_ok=True)
res = subprocess.run(['cuobjdump', '--dump-resource-usage', so], capture_output=True, text=True)
open(os.path.join(diag, 'cuda_shared_res_usage.txt'), 'w').write(res.stdout + res.stderr)
sass = subprocess.run(['cuobjdump', '-sass', so], capture_output=True, text=True)
open(os.path.join(diag, 'cuda_shared_sass.txt'), 'w').write(sass.stdout + sass.stderr)
ops = Counter()
for ln in sass.stdout.splitlines():
    m = re.search(r'/\*[0-9a-f]+\*/\s+(@!?P\d+\s+)?([A-Z][A-Z0-9_.]*)', ln)
    if m: ops[m.group(2).split('.')[0]] += 1
hmma = {k: v for k, v in ops.items() if 'HMMA' in k or 'IMMA' in k or 'MMA' in k}
print('resource usage head:'); print('\n'.join(res.stdout.splitlines()[:20]))
print('\nMMA-family opcodes :', hmma if hmma else 'NONE FOUND')
print('FFMA / LDL / STL   :', ops.get('FFMA',0), '/', ops.get('LDL',0), '/', ops.get('STL',0))
print('top opcodes        :', ops.most_common(12))
print('saved to', diag)

## Cell 19 — Shared vs naive vs Triton vs PyTorch: performance
Interleaved, warmed-up, synchronized timing with preallocated FP16 output on all four paths; captures SM clock per repeat. Ratio columns are > 1 when the shared kernel is faster. Writes `results/diagnostics/cuda/cuda_shared_performance.json`. No claims until reviewed.

In [ ]:
diag = os.path.join(REPO_ROOT, 'results', 'diagnostics', 'cuda')
rows = rws.performance(naive_lib, shared_lib, sizes=(256, 1024), repeats=5, out_dir=diag)
rws.print_performance(rows)